# الدرس 02 - استكشاف إطار عمل Microsoft Agent

يوفر **إطار عمل Microsoft Agent (MAF)** إطاراً موحداً لبناء وكيل ذكاء اصطناعي. يقدم بنية واضحة وقابلة للتكوين تتألف من أربعة مكونات أساسية:

- **العميل** – يتصل بنقطة نهاية نموذج الذكاء الاصطناعي ويتولى الاتصال
- **الوكيل** – يلف العميل بالتعليمات وتعريفات الأدوات
- **الأدوات** – توسع قدرات الوكيل بوظائف مخصصة يمكن للنموذج استدعاؤها
- **الجلسة** – يحافظ على سجل المحادثة للتفاعلات متعددة الأدوار

في هذا الدرس، سنبني **وكيل حجز سفر** يتحقق من توفر الوجهة باستخدام هذه المفاهيم.


## الإعداد


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## فهم بنية إطار الوكيل

يتبع إطار عمل الوكيل من مايكروسوفت بنية متعددة الطبقات:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **العميل** – يتصل `FoundryChatClient` بنشر Azure OpenAI. يدير المصادقة، وتنسيق الطلب، وتحليل الاستجابة.
2. **الوكيل** – يتم إنشاؤه من العميل عبر `provider.create_agent()`، يجمع الوكيل بين الوصول إلى النموذج والتعليمات (موجه النظام) والأدوات.
3. **الأدوات** – دوال بايثون مزينة بـ `@tool` يمكن للوكيل استدعاؤها لتنفيذ إجراءات أو استرجاع بيانات.
4. **الجلسة** – كائن `AgentSession` (يتم إنشاؤه عبر `agent.create_session()`) يخزن تاريخ المحادثة، مما يتيح الحوار متعدد الخطوات حيث يتذكر الوكيل السياق السابق.

دعونا نبني كل طبقة خطوة بخطوة.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## إضافة الأدوات باستخدام المزخرف @tool

تسمح الأدوات للوكيل باتخاذ إجراءات تتجاوز توليد النصوص. يقوم مزخرف `@tool` بتحويل دالة بايثون عادية إلى شيء يمكن للوكيل استدعاؤه.

نقاط رئيسية:
- استخدم `Annotated[type, "الوصف"]` لكي يفهم النموذج كل معلمة.
- تصبح الوثيقة النصية وصف الأداة الذي يراه النموذج.
- `approval_mode="never_require"` يعني أن الأداة تعمل تلقائيًا بدون تأكيد من المستخدم.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## إنشاء وكيل باستخدام الأدوات

الآن نقوم بدمج العميل، التعليمات، والأدوات في وكيل واحد. تعمل `instructions` كإرشادات النظام — فهي تحدد شخصية وسلوك الوكيل.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## محادثات متعددة الجولات مع الجلسات

يحتفظ `AgentSession` (الذي يتم إنشاؤه عبر `agent.create_session()`) بسجل لجميع الرسائل في المحادثة. من خلال تمرير نفس الجلسة إلى كل استدعاء لـ `agent.run()`، يمكن للوكيل الوصول إلى كامل تاريخ المحادثة والرجوع إلى الرسائل السابقة.

نمرر `tools=[check_destination_availability]` لكي يتمكن الوكيل من استخدام فاحص التوفر الخاص بنا خلال كل دورة.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## الملخص

في هذا الدرس استكشفت الركائز الأربع لإطار عمل وكيل مايكروسوفت:

| المفهوم | ما تعلمته |
|---------|------------------|
| **العميل** | `FoundryChatClient` يتصل بـ Azure OpenAI باستخدام مصادقة مستندة إلى بيانات الاعتماد |
| **الوكيل** | `provider.create_agent()` يجمع بين اتصال النموذج والتعليمات والاسم |
| **الأدوات** | الديكوريات `@tool` تكشف عن وظائف بايثون يمكن للوكيل استدعاؤها |
| **الجلسة** | `agent.create_session()` يحفظ سجل المحادثة عبر عدة دورات |

تتجمع هذه اللبنات الأساسية معًا لإنشاء وكلاء قادرين على إجراء محادثات طبيعية، واستدعاء وظائف خارجية، والحفاظ على السياق — وهو الأساس لأنماط الوكلاء المتقدمة في الدروس التالية.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
